# Run the restoration website on a Colab GPU

Runs the same app as on the laptop (`app/main.py` serves both the page and `POST /restore`), but the three models run on Colab's GPU. An ngrok tunnel gives it a public URL. The page and the API come from the same server, so one URL is all you need: no CORS setup and no change to `static/app.js`.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** (or any GPU).
2. Make a free ngrok account at https://ngrok.com, copy your authtoken from its dashboard, and add it in Colab's **Secrets** panel (key icon, left sidebar) under the name `NGROK_AUTHTOKEN`, with notebook access switched on.

Then run the cells top to bottom. Keep the laptop (CPU) setup as a fallback: free Colab GPUs are not guaranteed, and the URL changes every time the tunnel restarts.

In [ ]:
# 1. Check a GPU is attached. If this fails, change the runtime type (see above).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code (the repo root contains app/, static/, scripts/). Re-running pulls the latest main.
import os
REPO = '/content/Image-Restoration-Low-Latency'
if not os.path.exists(REPO):
    !git clone https://github.com/NOTtaLEMON/Image-Restoration-Low-Latency.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}

In [ ]:
# 3. Install only what Colab lacks. Not `pip install -r requirements.txt`: its version floors
#    (written for the laptop) would replace Colab's CUDA build of torch and its numpy.
!pip install -q fastapi uvicorn python-multipart huggingface_hub pyngrok

In [ ]:
# 4. Download the weights into hf_weights/ (pinned revision, SHA-256 checked).
!python scripts/download_weights.py

In [ ]:
# 5. Confirm the app will use the GPU. app/config.py picks cuda automatically when it is available.
!python -c "from app.config import DEVICE, IRSDE_BATCH_SIZE; print('DEVICE =', DEVICE, '| IRSDE_BATCH_SIZE =', IRSDE_BATCH_SIZE)"

In [ ]:
# 6. Start the server in the background and wait until the models have loaded.
import subprocess, time, urllib.request

!pkill -f 'uvicorn app.main:app' || true
log_file = open('server.log', 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000'],
    stdout=log_file, stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/', timeout=2)
        print('Server is up.')
        break
    except Exception:
        if server.poll() is not None:
            print('Server exited. Log:')
            break
        time.sleep(2)
else:
    print('Still not up after 4 minutes. Log so far:')
!tail -n 20 server.log

In [ ]:
# 7. Open a public URL with ngrok (reads the NGROK_AUTHTOKEN secret).
#    The first visit may show an ngrok warning page: click 'Visit Site' once.
from google.colab import userdata
from pyngrok import conf, ngrok

# Pass the token through pyngrok's config, not ngrok.set_auth_token(): re-running that fails with
# 'Authtoken already set in configuration file' once the token has been saved.
conf.get_default().auth_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.kill()  # close a tunnel left over from an earlier run
tunnel = ngrok.connect(8000, 'http')
print('Open this in the laptop browser:', tunnel.public_url)

In [ ]:
# 8. Server log, including per-stage timings for each request
#    ("NAFNet x s, UMapNet y s" and "IR-SDE: n of m patches in z s"). Re-run after each upload.
!tail -n 30 server.log

In [ ]:
# 9. Stop everything when done.
ngrok.kill()
server.terminate()
print('Stopped.')